In [2]:
pip install -U sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.8/570.8 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.3.0
    Uninstalling sentence-transformers-5.3.0:
      Successfully uninstalled sentence-transformers-5.3.0


In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("multi-qa-distilbert-cos-v1")
# smaller model built for semantic search + QA, smaller faster BERT



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [42]:
import pandas as pd

df = pd.read_csv('ArticlesforRTI.csv')

documents = df['text'].tolist()
titles = df['title'].tolist()
df

,article_id,disease_id,disease_name,title,url,text,symptoms,risk_factors,age_groups,smoking,bmi,comorbidities
0,ART_001,RTI_001,Pneumonia,Pneumonia Overview,https://www.who.int/news-room/fact-sheets/deta...,Pneumonia is an acute infection of the lung pa...,cough; purulent sputum; fever; dyspnoea,elderly; smoking; chronic disease,child; adult; elderly,current_smoker; ex_smoker,obese; underweight,COPD; asthma; heart failure; diabetes
1,ART_002,RTI_002,Influenza,Influenza Overview,https://www.who.int/news-room/fact-sheets/deta...,Influenza is a viral respiratory infection cau...,fever; cough; fatigue; myalgia,elderly; immunosuppressed,child; adult; elderly,current_smoker; ex_smoker,obese,asthma; COPD; diabetes
2,ART_003,RTI_003,Common Cold,Common Cold Overview,https://www.nhs.uk/conditions/common-cold/,Common cold is a mild viral infection affectin...,sneezing; sore throat; congestion,crowding; smoking,child; adult; elderly,current_smoker,all,asthma; immunosuppression
3,ART_004,RTI_004,Acute Sinusitis,Sinusitis Overview,https://www.nhs.uk/conditions/sinusitis/,Sinusitis is inflammation of sinuses causing f...,facial pain; congestion; nasal discharge,allergies; smoking,child; adult; elderly,current_smoker,all,asthma; allergic rhinitis
4,ART_005,RTI_005,Pharyngitis and Strep Throat,Strep Throat Overview,https://www.cdc.gov/groupastrep/diseases-publi...,Pharyngitis is inflammation of the throat comm...,sore throat; fever; painful swallowing,close contact; school exposure,child; adult,current_smoker,all,diabetes; immunosuppression
5,ART_006,RTI_006,Laryngitis and Croup,Laryngitis Overview,https://www.nhs.uk/conditions/laryngitis/,Laryngitis involves inflammation of the larynx...,hoarseness; barking cough; stridor,viral infection; children,child; adult,non_smoker,all,none
6,ART_007,RTI_007,Acute Bronchitis,Bronchitis Overview,https://www.nhs.uk/conditions/bronchitis/,Bronchitis is inflammation of bronchial tubes ...,cough; sputum; fatigue,smoking; air pollution,adult; elderly,current_smoker; ex_smoker,all,COPD; asthma
7,ART_008,RTI_008,Bronchiolitis,Bronchiolitis Overview,https://www.nhs.uk/conditions/bronchiolitis/,Bronchiolitis is a viral infection affecting i...,wheezing; cough; feeding difficulty,infants; prematurity,infant,passive_smoker,all,prematurity
8,ART_009,RTI_009,COPD Exacerbation,COPD Exacerbation Overview,https://www.nhs.uk/conditions/chronic-obstruct...,COPD exacerbation is worsening of respiratory ...,increased dyspnoea; sputum; cough,smoking; infections,adult; elderly,current_smoker; ex_smoker,underweight,COPD; heart disease
9,ART_010,RTI_010,Tuberculosis,Tuberculosis Overview,https://www.who.int/news-room/fact-sheets/deta...,Tuberculosis is a bacterial infection affectin...,chronic cough; night sweats; weight loss,crowding; HIV,adult; elderly,current_smoker,underweight,HIV; diabetes


In [5]:
from sentence_transformers import util
import torch

# Encode and save
doc_embeddings = model.encode(documents, convert_to_tensor=True)  # key change: tensor instead of numpy
torch.save(doc_embeddings, 'doc_embeddings.pt')

# Load embeddings
doc_embeddings = torch.load('doc_embeddings.pt')


In [6]:
age_groups = ["child", "adult", "elderly"]
bmi = ["all", "obese", "underweight"]
comorbidities = ["COPD", "HIV", "allergic rhinitis", "asthma", "chronic lung disease",
                 "diabetes", "heart disease", "heart failure", "hypertension",
                 "immunosuppression", "none", "prematurity"]
smoking = ['current_smoker', "ex_smoker", "passive_smoker","non_smoker"]

def total_score(semantic_score, dem_score):
    return semantic_score * dem_score

In [31]:
def one_hot_encode(value, feature_list):
    """
    Encode a value as one-hot vector against feature list.
    Example: one_hot_encode('child', age_groups) -> [1, 0, 0]
    """
    return [1 if val == value else 0 for val in feature_list]


def dem_score(user_profile, doc_row):

    # Age dimension: user has one age, doc has multiple
    user_age_vec = one_hot_encode(user_profile['age_group'], age_groups)
    doc_age_vec = [1 if age in doc_row['age_groups'] else 0 for age in age_groups]
    age_score = np.dot(user_age_vec, doc_age_vec) / len(age_groups)

    # Smoking dimension
    user_smoking_vec = one_hot_encode(user_profile['smoking'], smoking)
    doc_smoking_vec = [1 if smk in doc_row['smoking'] else 0 for smk in smoking]
    smoking_score = np.dot(user_smoking_vec, doc_smoking_vec) / len(smoking)

    # BMI dimension
    user_bmi_vec = one_hot_encode(user_profile['bmi'], bmi)
    doc_bmi_vec = [1 if b in doc_row['bmi'] else 0 for b in bmi]
    bmi_score = np.dot(user_bmi_vec, doc_bmi_vec) / len(bmi)

    # Comorbidities dimension
    user_comorbidities_vec = one_hot_encode(user_profile['comorbidities'], comorbidities)
    doc_comorbidities_vec = [1 if com in doc_row['comorbidities'] else 0 for com in comorbidities]
    comorb_score = np.dot(user_comorbidities_vec, doc_comorbidities_vec) / len(comorbidities)

    # Average across all dimensions for final score
    final_score = (age_score + smoking_score + bmi_score + comorb_score) / 4.0

    return final_score

In [34]:
def retrieve(query, user_profile, top_k=10, semantic_weight=0.5):
  #  0.5 = RECOMMENDED for medical/health contexts.
  # Demographics are as important as semantic relevance because
  #medical advice must be personalized to the patient's risk profile.

  #Use 0.7+ for general search (prioritize relevance)
  #Use 0.3- for strict personalization (prioritize demographics)

    # Get semantic search results
    query_embedding = model.encode(query, convert_to_tensor=True)
    results = util.semantic_search(query_embedding, doc_embeddings, top_k=top_k*2)
    results = results[0]

    # Score each result
    scored_docs = []
    for result in results:
        idx = result['corpus_id']
        semantic_score = result['score']
        demographic_score = dem_score(user_profile, df.iloc[idx])

        # Hybrid score: weighted combination of semantic and demographic
        final_score = (semantic_weight * semantic_score) + \
                      ((1 - semantic_weight) * demographic_score)

        scored_docs.append({
            'idx': idx,
            'semantic_score': semantic_score,
            'demographic_score': demographic_score,
            'final_score': final_score,
            'title': titles[idx],
            'text': documents[idx][:100]
        })

    # Sort by final score (descending)
    scored_docs.sort(key=lambda x: x['final_score'], reverse=True)

    # Display results
    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"User Profile: {user_profile}")
    print(f"Semantic Weight: {semantic_weight*100:.0f}% | Demographic Weight: {(1-semantic_weight)*100:.0f}%")
    print(f"{'='*70}\n")

    for i, doc in enumerate(scored_docs[:top_k], 1):
        print(f"{i}. [{doc['final_score']:.3f}] {doc['title']}")
        print(f"   └─ Semantic: {doc['semantic_score']:.3f} | Demographic: {doc['demographic_score']:.3f}")
        print(f"   └─ {doc['text']}...\n")

    return scored_docs[:top_k]


In [50]:
user_profile = {
    'age_group': 'child',
    'smoking': 'non_smoker',
    'bmi': 'underweight',
    'comorbidities': 'asthma'
}

retrieve("shortness of breath", user_profile, top_k=3, semantic_weight=0.5)
retrieve("coughing up blood", user_profile, top_k=3, semantic_weight=0.5)
retrieve("baby struggling to breathe", user_profile, top_k=3, semantic_weight=1.0)

In [44]:
from sklearn.feature_extraction.text import CountVectorizer

#BM25 preprocessing
def setup_bm25(documents):

    vectorizer = CountVectorizer(
        stop_words='english',
        lowercase=True,
        token_pattern=r'(?u)\b\w\w+\b'  # Min 2 character words
    )

    term_doc_mat = vectorizer.fit_transform(documents)
    vocabulary = vectorizer.get_feature_names_out()
    tfs = term_doc_mat.toarray()

    # BM25 parameters
    dls = np.array([len(doc.split()) for doc in documents])
    avgdl = np.mean(dls)
    k1 = 1.2
    b = 0.8

    # IDF calculation
    df_counts = np.sum(tfs > 0, axis=0)
    idf = np.log((len(documents) - df_counts + 0.5) / (df_counts + 0.5) + 1)

    # BM25 scores
    denominator = tfs + k1 * (1 - b + b * (dls.reshape(-1, 1) / avgdl))
    numerator = (k1 + 1) * tfs
    bm25_tf = numerator / denominator
    bm25_scores = idf * bm25_tf

    return {
        'vectorizer': vectorizer,
        'vocabulary': vocabulary,
        'bm25_scores': bm25_scores,
        'vocab_dict': {word: idx for idx, word in enumerate(vocabulary)}
    }


In [47]:
def bm25_retrieve(query, user_profile, bm25_data, top_k=10, semantic_weight=0.5):
    # Preprocess query (lowercase, split)
    query_words = query.lower().split()
    vocab_dict = bm25_data['vocab_dict']

    # Find matching vocabulary indices
    query_indices = []
    for word in query_words:
        if word in vocab_dict:
            query_indices.append(vocab_dict[word])

    # If NO query terms match vocabulary, fall back to demographic filtering
    if not query_indices:
        print(f"⚠️ WARNING: Query '{query}' has no matching terms in vocabulary")
        print("Falling back to demographic filtering...\n")

        scored_docs = []
        for doc_idx in range(len(documents)):
            dem_score_val = dem_score(user_profile, df.iloc[doc_idx])
            scored_docs.append({
                'idx': doc_idx,
                'bm25_score': 0.0,
                'demographic_score': dem_score_val,
                'final_score': dem_score_val,
                'title': titles[doc_idx],
                'text': documents[doc_idx][:100] # Added 'text' field here
            })

        scored_docs.sort(key=lambda x: x['final_score'], reverse=True)
        return scored_docs[:top_k]

    # Calculate BM25 scores for matching terms
    bm25_scores = bm25_data['bm25_scores']
    doc_bm25_scores = bm25_scores[:, query_indices].sum(axis=1)

    # Normalize BM25 scores to [0, 1]
    max_bm25 = np.max(doc_bm25_scores)
    if max_bm25 > 0:
        normalized_bm25 = doc_bm25_scores / max_bm25
    else:
        normalized_bm25 = doc_bm25_scores

    # Score each document
    scored_docs = []
    for doc_idx in range(len(documents)):
        bm25_norm = float(normalized_bm25[doc_idx])
        dem_score_val = dem_score(user_profile, df.iloc[doc_idx])

        # Hybrid score
        final_score = (semantic_weight * bm25_norm) + \
                      ((1 - semantic_weight) * dem_score_val)

        scored_docs.append({
            'idx': doc_idx,
            'bm25_score': bm25_norm,
            'demographic_score': dem_score_val,
            'final_score': final_score,
            'title': titles[doc_idx],
            'text': documents[doc_idx][:100]
        })

    # Sort by final score
    scored_docs.sort(key=lambda x: (-x['final_score'], doc_idx))

    return scored_docs[:top_k]

In [49]:
# Evaluation Framework

ground_truths = {
    "pneumonia symptoms": [0],
    "shortness of breath": [0, 8, 13],
    "coughing up blood": [14],
    "loss of smell": [10],
    "baby struggling to breathe": [7, 11]
}

def evaluate_retriever(queries_dict, user_profile, retriever_func, top_k=3):

   # Evaluate retriever and return precision score.

    total_precision = 0

    for query, expected_ids in queries_dict.items():
        results = retriever_func(query, user_profile, top_k)

        # Extract IDs from results
        retrieved_ids = [res['idx'] for res in results]

        # Calculate precision
        relevant = len(set(retrieved_ids) & set(expected_ids))
        precision = relevant / top_k
        total_precision += precision

    return total_precision / len(queries_dict)

if __name__ == "__main__":

    user_profile = {
        'age_group': 'child',
        'smoking': 'non_smoker',
        'bmi': 'underweight',
        'comorbidities': 'asthma'
    }

    sbert_precision = evaluate_retriever(
        ground_truths, user_profile,
        retriever_func=retrieve,
        top_k=3
    )

    bm25_precision = evaluate_retriever(
        ground_truths, user_profile,
        retriever_func=lambda q, up, k: bm25_retrieve(q, up, bm25_data, k),
        top_k=3
    )

    print(f"SBERT Precision@3: {sbert_precision:.2f}")
    print(f"BM25 Precision@3:  {bm25_precision:.2f}")


Query: 'pneumonia symptoms'
User Profile: {'age_group': 'child', 'smoking': 'non_smoker', 'bmi': 'underweight', 'comorbidities': 'asthma'}
Semantic Weight: 50% | Demographic Weight: 50%

1. [0.420] Pneumonia Overview
   └─ Semantic: 0.653 | Demographic: 0.188
   └─ Pneumonia is an acute infection of the lung parenchyma causing cough fever dyspnoea and chest pain....

2. [0.293] Influenza Overview
   └─ Semantic: 0.483 | Demographic: 0.104
   └─ Influenza is a viral respiratory infection causing fever fatigue cough and systemic illness....

3. [0.266] Pertussis Overview
   └─ Semantic: 0.387 | Demographic: 0.146
   └─ Pertussis is a highly contagious bacterial infection causing severe coughing fits....


Query: 'shortness of breath'
User Profile: {'age_group': 'child', 'smoking': 'non_smoker', 'bmi': 'underweight', 'comorbidities': 'asthma'}
Semantic Weight: 50% | Demographic Weight: 50%

1. [0.216] Laryngitis Overview
   └─ Semantic: 0.285 | Demographic: 0.146
   └─ Laryngitis involve

In [20]:
pip install gradio

In [52]:
import gradio as gr

#GUI

def search_interface(query, age_group, smoking, bmi, comorbidities, retriever_type, top_k):
    """
    Main search function for GUI.
    """
    user_profile = {
        'age_group': age_group,
        'smoking': smoking,
        'bmi': bmi,
        'comorbidities': comorbidities
    }

    # Choose retriever
    if retriever_type == "SBERT (Semantic)":
        results = retrieve(query, user_profile, top_k=int(top_k))
    else:
        results = bm25_retrieve(query, user_profile, bm25_data, top_k=int(top_k))

    # Format output
    output = []
    for i, res in enumerate(results, 1):
        output.append(f"**{i}. {res['title']}** (Score: {res['final_score']:.3f})\n")
        output.append(f"Semantic: {res.get('semantic_score', res.get('bm25_score', 0)):.3f} | Demographic: {res['demographic_score']:.3f}\n")
        output.append(f"{res['text']}...\n\n")

    return "".join(output) if output else "No results found."


#interface

with gr.Blocks(title="RTI Search Engine") as demo:
    gr.Markdown("# RTI (Respiratory Tract Infection) Search Engine")
    gr.Markdown("Search for medical articles personalized to your profile.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Your Profile")
            age_group = gr.Dropdown(
                choices=["child", "adult", "elderly"],
                value="child",
                label="Age Group"
            )
            smoking = gr.Dropdown(
                choices=["current_smoker", "ex_smoker", "passive_smoker", "non_smoker"],
                value="non_smoker",
                label="Smoking Status"
            )
            bmi = gr.Dropdown(
                choices=["normal", "obese", "underweight"],
                value="underweight",
                label="BMI"
            )
            comorbidities = gr.Dropdown(
                choices=["COPD", "HIV", "asthma", "diabetes", "heart disease",
                         "heart failure", "hypertension", "immunosuppression", "none"],
                value="asthma",
                label="Comorbidities"
            )

        with gr.Column():
            gr.Markdown("### Search Settings")
            query = gr.Textbox(
                label="Search Query",
                placeholder="e.g., shortness of breath",
                lines=3
            )
            retriever_type = gr.Radio(
                choices=["SBERT (Semantic)", "BM25 (Keyword)"],
                value="SBERT (Semantic)",
                label="Retrieval Method"
            )
            top_k = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Number of Results"
            )

    search_button = gr.Button("Search", variant="primary")

    output = gr.Markdown(label="Results")

    # Connect button
    search_button.click(
        fn=search_interface,
        inputs=[query, age_group, smoking, bmi, comorbidities, retriever_type, top_k],
        outputs=output
    )

if __name__ == "__main__":
    demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://18004f8d11bd7b51ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
